# 🏗 Анализ рынка недвижимости Турции
### 👥 Авторы: Тимур Еникеев и Гареев Рустам
📅 **Дата:** 2026

---
**🎯 Цель проекта:** 
Провести полный разведочный анализ данных (EDA), очистить датасет, сгенерировать новые признаки и подготовить данные к обучению регрессионных моделей для прогнозирования стоимости недвижимости.

**📑 Этапы работы:**
1. ️‍♂️ **EDA** — исследование структуры, распределений, пропусков, корреляций
2.  **Предобработка** — очистка, кодирование, масштабирование, Feature Engineering
3. 🤖 **Моделирование** — обучение, подбор гиперпараметров, выбор лучшей модели
4.  **Применение** — сохранение модели для деплоя в приложение

## 📖 Источник данных

Датасет предоставлен турецкой платформой недвижимости **Zingat.com** 
(https://www.zingat.com) — крупнейший портал по продаже и аренде 
недвижимости в Турции.

**Период сбора:** 2018–2019 гг.  
**Объём:** ~403 000 объявлений  
**География:** вся Турция (Стамбул, Анкара, Измир и др.)  
**Целевая переменная:** `price` — цена объекта в турецких лирах (TRY)

Данные получены в рамках соревнования по прогнозированию стоимости 
недвижимости. Словарь признаков описан в файле `zingat_data_info.docx`.


### 📍 Локация
*   **`city`** — Город, в котором находится объект (например, Стамбул, Анкара, Измир).
*   **`district`** — Район города.
*   **`neighborhood`** — Микрорайон или квартал.
*   *(Примечание: эти три признака обычно извлекаются из полного адреса `address`, который сам по себе в модель не передается из-за слишком большого количества уникальных значений).*

### 🏠 Характеристики объекта
*   **`type`** — Глобальный тип недвижимости (в данном датасете почти везде "Жильё").
*   **`sub_type`** — Конкретный подтип недвижимости (Квартира, Вилла, Отдельный дом, Резиденция, Дача и т.д.).
*   **`room_count`** — Планировка/количество комнат в турецком формате (например, "3+1" означает 3 спальни + 1 гостиная). *В процессе обработки обычно преобразуется в числовой признак `rooms_numeric` (сумма цифр, т.е. 4).*
*   **`size`** — Общая площадь объекта в квадратных метрах (м²).
*   **`building_age`** — Возраст здания в годах. Может быть указан как конкретное число или диапазон (например, "0", "6-10", "20 и более"). *Преобразуется в числовое значение (середина диапазона или граничное значение).*
*   **`total_floor_count`** — Общее количество этажей в здании.
*   **`floor_no`** — Номер этажа, на котором находится конкретный объект. Может содержать специфические значения, такие как "Zemin kat" (цокольный/первый этаж) или "Çatı Katı" (мансардный/последний этаж).
*   **`heating_type`** — Тип системы отопления (например, Kombi/Doğalgaz — индивидуальный газовый котел, Klima — кондиционер, Merkezi Sistem — центральное отопление, Soba — печное).

### 💰 Финансы и сделка
*   **`listing_type`** — Тип сделки или размещения (Продажа, Долгосрочная аренда, Аренда на день/посуточно).
*   **`price`** — Стоимость объекта. Является **целевой переменной** (то, что мы предсказываем). *В процессе обработки преобразуется в `log_price` (логарифм цены) для улучшения работы моделей.*
*   **`price_currency`** — Валюта, в которой указана цена (в основном TRY — турецкая лира, иногда встречается EUR или USD).

### 📅 Временные метки и статистика
*   **`id`** — Уникальный идентификатор объявления (удаляется или не используется при обучении модели, так как не несет смысловой нагрузки).
*   **`start_date`** — Дата размещения объявления на сайте.
*   **`end_date`** — Дата снятия объявления с публикации (может быть пустым, если объявление еще активно).
*   **`tom`** (Time On Market) — Срок экспозиции. Количество дней, которое объявление провисело на сайте до снятия или до момента выгрузки данных.

---

# Прогнозирование стоимости недвижимости в Турции: полный цикл анализа данных

## 1. Общие сведения о проекте

Настоящий проект посвящён построению регрессионной модели для прогнозирования стоимости объектов недвижимости на рынке Турции. Работа выполнена в рамках учебного задания и охватывает полный цикл анализа данных: от первичного исследования (EDA) до построения, валидации и сохранения финальной модели машинного обучения.

**Актуальность задачи.** Рынок недвижимости Турции характеризуется высокой волатильностью цен, значительной региональной дифференциацией и большим количеством факторов, влияющих на стоимость объекта. Автоматизированное прогнозирование стоимости позволяет:
- снизить информационную асимметрию между продавцами и покупателями;
- повысить эффективность работы риелторских агентств;
- обеспечить аналитическую поддержку для инвестиционных решений;
- создать основу для автоматизированных систем оценки (AVM — Automated Valuation Models).

**Целевая переменная.** В качестве целевой переменной выступает `price` — цена объекта в турецких лирах (TRY). Переменная имеет ярко выраженную правостороннюю асимметрию, что требует применения логарифмического преобразования перед обучением моделей.

**Источник данных.** Датасет предоставлен турецкой платформой недвижимости Zingat.com и содержит информацию о 403 487 объявлениях за период 2018–2019 годов. Данные охватывают всю территорию Турции и включают как объекты продажи, так и объекты аренды.

## 2. Структура проекта

Проект структурирован в соответствии с требованиями технического задания и включает четыре последовательных этапа:

| Этап | Содержание | Результат |
|------|-----------|-----------|
| 1. Исследование данных (EDA) | Анализ структуры, распределений, пропусков, корреляций | Сформулированные гипотезы |
| 2. Предварительная обработка | Очистка, кодирование, Feature Engineering, масштабирование | Готовый к моделированию датасет |
| 3. Построение моделей | Обучение, подбор гиперпараметров, валидация | Финальная модель (Random Forest) |
| 4. Применение | Сохранение модели, подготовка к деплою | Файлы `best_model_pipeline.pkl`, `feature_schema.pkl` |

## 3. Технологический стек

- **Язык программирования:** Python 3.13
- **Среда выполнения:** Jupyter Notebook
- **Библиотеки анализа данных:** pandas, numpy, scipy, statsmodels
- **Библиотеки визуализации:** matplotlib, seaborn
- **Библиотеки машинного обучения:** scikit-learn
- **Сохранение моделей:** joblib

## 4. Ключевые методологические решения

1. **Логарифмирование целевой переменной.** Исходное распределение цены имеет экстремальную асимметрию (skew ≈ 308). Применение преобразования `log1p` снижает асимметрию до −0.81, что делает распределение близким к нормальному и существенно улучшает качество линейных моделей.

2. **Разбиение данных 70/15/15.** Данные разделены на обучающую (70%), валидационную (15%) и тестовую (15%) выборки с фиксацией случайного состояния (`random_state=42`) для воспроизводимости результатов.

3. **Обучение препроцессинга только на train.** Все трансформации (импутация, масштабирование, One-Hot Encoding) обучаются исключительно на обучающей выборке, что исключает утечку данных (data leakage).

4. **Выбор Random Forest в качестве финальной модели.** По результатам сравнения трёх моделей разной природы (Ridge, Decision Tree, Random Forest) модель Random Forest демонстрирует наилучшее качество на валидационной выборке (MAE_log = 0.241, R²_log = 0.974).

---

## Выводы по введению

Проект построен на методологически корректной основе: соблюдены принципы разделения данных, исключена утечка информации, применено адекватное преобразование целевой переменной. Выбор трёх моделей разной природы (линейная, дерево, ансамбль) позволяет провести содержательное сравнение и обосновать выбор финального решения.

## 📦 1. Импорт библиотек и настройка окружения

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import re
import joblib
import warnings
from IPython.display import display
from scipy.stats import skew, kurtosis
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')

sns.set_theme(style="darkgrid", palette="Set2")
plt.rcParams['figure.figsize'] = (11, 6)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['font.family'] = 'DejaVu Sans'

RANDOM_STATE = 42


In [ ]:
%%writefile real_estate_features.py
"""Признаки для модели стоимости недвижимости (Zingat, Турция).

Модуль вынесен из ноутбука, чтобы pickle-файл pipeline можно было загрузить
в приложении (Streamlit/FastAPI): положите этот файл рядом с приложением.
"""
import re

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin

# --------------------------------------------------------------------------
# Валюты. Цена в датасете указана в валюте объявления (TRY / EUR / USD ...).
# Курсы ПРИБЛИЗИТЕЛЬНЫЕ (средние за период сбора: осень 2018 - весна 2019).
# Неточность курса частично компенсируется тем, что price_currency
# дополнительно подаётся модели как признак.
# --------------------------------------------------------------------------
FX_TO_TRY = {'TRY': 1.0, 'TL': 1.0, 'USD': 5.5, 'EUR': 6.3, 'GBP': 7.0}


def to_try(price, currency, default_currency='TRY'):
    """Переводит цену в турецкие лиры. Неизвестная валюта -> NaN."""
    price = pd.to_numeric(pd.Series(price), errors='coerce')
    cur = pd.Series(currency, index=price.index).fillna(default_currency)
    cur = cur.astype(str).str.strip().str.upper()
    rate = cur.map(FX_TO_TRY)
    return price * rate


# --------------------------------------------------------------------------
# Парсеры текстовых значений
# --------------------------------------------------------------------------
_NUM = re.compile(r'\d+')
TOP_FLOOR = 9999.0  # маркер "последний/мансардный этаж"


def _nums(s):
    return [int(n) for n in _NUM.findall(s)]


def _is_number(x):
    return isinstance(x, (int, float, np.integer, np.floating)) and not isinstance(x, bool)


def parse_building_age(x):
    """'0' -> 0, '6-10 лет' -> 8, '21 и более' -> 21, '20 ve üzeri' -> 20."""
    if pd.isna(x):
        return np.nan
    if _is_number(x):
        return float(x)
    s = str(x).strip().lower()
    n = _nums(s)
    if len(n) >= 2:
        return (n[0] + n[1]) / 2.0
    if len(n) == 1:
        return float(n[0])
    if any(k in s for k in ('нов', 'yeni', 'sıfır', 'строя', 'inşaat')):
        return 0.0
    return np.nan


def parse_total_floors(x):
    """'20 и более' -> 20, '5' -> 5."""
    if pd.isna(x):
        return np.nan
    if _is_number(x):
        return float(x)
    n = _nums(str(x))
    return float(n[0]) if n else np.nan


def parse_floor_no(x):
    """Номер этажа. Подвал -> -1, цоколь/вход -> 0, высокий первый -> 1,
    мансарда/крыша -> TOP_FLOOR (позже заменяется на total_floor_count)."""
    if pd.isna(x):
        return np.nan
    if _is_number(x):
        return float(x)
    s = str(x).strip().lower()
    if any(k in s for k in ('подвал', 'bodrum', 'kot', 'полуподвал')):
        return -1.0
    if any(k in s for k in ('мансард', 'çatı', 'cati', 'крыш', 'чердак')):
        return TOP_FLOOR
    if any(k in s for k in ('высок', 'yüksek', 'yuksek')):
        return 1.0
    if any(k in s for k in ('цокол', 'zemin', 'giriş', 'giris', 'входн', 'земл', 'сад', 'bahçe', 'bahce')):
        return 0.0
    n = _nums(s)
    return float(n[0]) if n else np.nan


def parse_rooms(x):
    """'3+1' -> 4 (сумма цифр)."""
    if pd.isna(x):
        return np.nan
    n = _nums(str(x))
    return float(sum(n)) if n else np.nan


def parse_bedrooms(x):
    """'3+1' -> 3 (первая цифра)."""
    if pd.isna(x):
        return np.nan
    n = _nums(str(x))
    return float(n[0]) if n else np.nan


def map_unique(series, func):
    """Применяет func только к уникальным значениям (быстро на 400k строк)."""
    uniq = pd.Series(series.dropna().unique())
    mapping = {u: func(u) for u in uniq}
    return series.map(mapping).astype(float)


def _col(X, name):
    return X[name] if name in X.columns else pd.Series(np.nan, index=X.index, dtype=object)


# --------------------------------------------------------------------------
# Главный трансформер: сырые столбцы объявления -> модельные признаки
# --------------------------------------------------------------------------
RAW_FEATURES = ['sub_type', 'start_date', 'listing_type', 'building_age',
                'total_floor_count', 'floor_no', 'room_count', 'size',
                'heating_type', 'price_currency', 'city', 'district',
                'neighborhood']

NUM_COLS = ['size', 'log_size', 'size_missing', 'size_per_room',
            'building_age', 'building_age_missing', 'is_new_building',
            'total_floor_count', 'floor_no', 'floor_ratio',
            'is_top_floor', 'is_ground_floor',
            'rooms_numeric', 'bedrooms', 'date_num', 'month',
            'district_freq', 'neigh_freq']
LOWCARD_COLS = ['listing_type', 'sub_type', 'heating_type',
                'price_currency', 'room_count']
# Географические ключи включают listing_type: уровень цен продажи и аренды
# отличается в ~100 раз, поэтому кодируем их отдельно.
HIGHCARD_COLS = ['city_lt', 'district_lt', 'neigh_lt']


class RealEstateFeatures(BaseEstimator, TransformerMixin):
    """Парсинг + заполнение площади + географические ключи.
    Всё, что считается по данным (медианы, частоты), обучается только в fit()."""

    SIZE_MIN, SIZE_MAX = 10.0, 1000.0
    DATE_ORIGIN = pd.Timestamp('2018-01-01')

    # -- служебное -------------------------------------------------------
    def _prepare(self, X):
        X = pd.DataFrame(X).copy()
        out = pd.DataFrame(index=X.index)

        size = pd.to_numeric(_col(X, 'size'), errors='coerce')
        size = size.where((size >= self.SIZE_MIN) & (size <= self.SIZE_MAX))
        out['size_raw'] = size
        out['rooms_numeric'] = map_unique(_col(X, 'room_count'), parse_rooms)
        out['bedrooms'] = map_unique(_col(X, 'room_count'), parse_bedrooms)

        for c in ['listing_type', 'sub_type', 'heating_type', 'price_currency',
                  'room_count', 'city', 'district', 'neighborhood']:
            out[c] = _col(X, c).astype('object').where(_col(X, c).notna(), 'Unknown').astype(str)

        out['building_age'] = map_unique(_col(X, 'building_age'), parse_building_age)
        out['total_floor_count'] = map_unique(_col(X, 'total_floor_count'), parse_total_floors)
        floor = map_unique(_col(X, 'floor_no'), parse_floor_no)
        is_top = floor == TOP_FLOOR
        floor = floor.where(~is_top, out['total_floor_count'])
        out['floor_no'] = floor
        out['is_top_floor'] = is_top.astype('float32')

        dt = pd.to_datetime(_col(X, 'start_date'), format='%m/%d/%y', errors='coerce')
        if dt.isna().all():
            dt = pd.to_datetime(_col(X, 'start_date'), errors='coerce')
        out['date_num'] = (dt - self.DATE_ORIGIN).dt.days.astype(float)
        out['month'] = dt.dt.month.astype(float)

        out['city_lt'] = out['listing_type'] + '|' + out['city']
        out['district_lt'] = out['city_lt'] + '|' + out['district']
        out['neigh_lt'] = out['district_lt'] + '|' + out['neighborhood']
        out['_district_key'] = out['city'] + '|' + out['district']
        out['_neigh_key'] = out['_district_key'] + '|' + out['neighborhood']
        out['_size_key'] = out['sub_type'] + '|' + out['rooms_numeric'].astype(str)
        return out

    # -- fit / transform -------------------------------------------------
    def fit(self, X, y=None):
        p = self._prepare(X)
        valid = p['size_raw'].notna()
        self.size_med_key_ = p.loc[valid].groupby('_size_key')['size_raw'].median().to_dict()
        self.size_med_rooms_ = p.loc[valid].groupby('rooms_numeric')['size_raw'].median().to_dict()
        self.size_med_global_ = float(p.loc[valid, 'size_raw'].median()) if valid.any() else 100.0
        self.district_cnt_ = p['_district_key'].value_counts().to_dict()
        self.neigh_cnt_ = p['_neigh_key'].value_counts().to_dict()
        self.n_train_ = len(p)
        return self

    def transform(self, X):
        p = self._prepare(X)
        size = p['size_raw']
        miss = size.isna()
        fill = p['_size_key'].map(self.size_med_key_)
        fill = fill.fillna(p['rooms_numeric'].map(self.size_med_rooms_))
        fill = fill.fillna(self.size_med_global_)
        p['size'] = size.fillna(fill)
        p['size_missing'] = miss.astype('float32')
        p['log_size'] = np.log1p(p['size'])
        p['size_per_room'] = p['size'] / p['rooms_numeric'].clip(lower=1)

        p['building_age_missing'] = p['building_age'].isna().astype('float32')
        p['is_new_building'] = (p['building_age'] <= 5).astype('float32').where(p['building_age'].notna(), np.nan)
        p['is_ground_floor'] = (p['floor_no'] <= 0).astype('float32').where(p['floor_no'].notna(), np.nan)
        p['floor_ratio'] = p['floor_no'] / p['total_floor_count'].replace(0, np.nan)

        p['district_freq'] = p['_district_key'].map(self.district_cnt_).fillna(0).astype(float) / self.n_train_
        p['neigh_freq'] = p['_neigh_key'].map(self.neigh_cnt_).fillna(0).astype(float) / self.n_train_
        return p[NUM_COLS + LOWCARD_COLS + HIGHCARD_COLS]


In [ ]:
# Модуль с признаками создан в предыдущей ячейке (%%writefile).
# Он нужен и ноутбуку, и приложению: pickle-файл pipeline ссылается на класс RealEstateFeatures.
import importlib
import real_estate_features
importlib.reload(real_estate_features)
from real_estate_features import (
    FX_TO_TRY, to_try, map_unique, TOP_FLOOR,
    parse_building_age, parse_total_floors, parse_floor_no, parse_rooms,
    RealEstateFeatures, RAW_FEATURES, NUM_COLS, LOWCARD_COLS, HIGHCARD_COLS
)
print("Модуль real_estate_features загружен.")


## 📂 2. Загрузка данных и первичный осмотр

In [ ]:
# Загрузка датасета
# Файл real_estate_data.csv ищем рядом с ноутбуком, в ./data и по старому пути.
from pathlib import Path

DATA_CANDIDATES = [
    Path("real_estate_data.csv"),
    Path("data") / "real_estate_data.csv",
    Path(r"C:\Users\Рустам\Desktop\turkey\real_estate_data.csv"),
]
DATA_PATH = next((p for p in DATA_CANDIDATES if p.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "Не найден real_estate_data.csv. Проверены пути:\n"
        + "\n".join(f"  {p.resolve()}" for p in DATA_CANDIDATES)
    )

df = pd.read_csv(DATA_PATH, low_memory=False)

# ВАЖНО: price указана в ВАЛЮТЕ ОБЪЯВЛЕНИЯ (TRY / EUR / USD ...). Складывать их вместе
# нельзя: 48 000 EUR выглядело бы в данных как дешёвая квартира. Переводим всё в TRY,
# исходную цену сохраняем отдельно, а price_currency остаётся признаком модели.
print("Валюты в датасете:")
print(df['price_currency'].value_counts(dropna=False))
df['price_original'] = df['price']
df['price'] = to_try(df['price_original'], df['price_currency'])
unknown_cur = df['price'].isna() & df['price_original'].notna()
print(f"\nСтрок с неизвестной валютой (цена -> NaN): {unknown_cur.sum()}")

print(df.columns.tolist())
print(f"Размерность: {df.shape[0]} строк x {df.shape[1]} столбцов")

# df.duplicated() с уникальным id всегда даёт 0 — сравниваем без служебных полей.
dup_cols = [c for c in df.columns if c not in ('id', 'tom', 'start_date', 'end_date', 'price_original')]
print(f"\nПолных дубликатов (с id): {df.duplicated().sum()}")
print(f"Дубликатов объявлений (без id/tom/дат): {df.duplicated(subset=dup_cols).sum()}")
print(f"\nТипы данных:")
print(df.dtypes.value_counts())
display(df.head(3))

print("Последние 3 строки:")
display(df.tail(3))

print("\nСлучайная строка:")
display(df.sample(1, random_state=RANDOM_STATE))

print("Уникальных значений по категориальным признакам:")
for col in ['city', 'district', 'neighborhood', 'listing_type', 'type', 'sub_type', 'heating_type', 'room_count']:
    if col in df.columns:
        print(f"  {col}: {df[col].nunique()} уникальных значений")


In [ ]:
# Описательная статистика числовых признаков
df.describe().T.style.format({'mean': '{:.1f}', 'std': '{:.1f}', 
                               'min': '{:.1f}', 'max': '{:.1f}',
                               '25%': '{:.1f}', '50%': '{:.1f}', '75%': '{:.1f}'})

In [ ]:
for col in ['city']:
    if col in df.columns:
        print(f"--- {col} ---")
        print(df[col].value_counts().head(8))
        print()


## 2.1 Первичный анализ структуры данных

Загруженный датасет содержит **403 487 строк и 18 столбцов**. Общий объём данных составляет приблизительно 400 тысяч наблюдений, что является достаточным для построения устойчивых регрессионных моделей.

**Типы признаков:**
- **Числовые (int64, float64):** 4 признака (`id`, `tom`, `size`, `price`)
- **Строковые (object):** 14 признаков, включая категориальные и текстовые

**Ключевые наблюдения:**

1. **Отсутствие полных дубликатов.** Проверка на дубликаты записей показала нулевое количество полных совпадений, что свидетельствует о качественной первичной очистке данных на стороне источника.

2. **Высокая кардинальность географических признаков.** Признак `neighborhood` (микрорайон) содержит 4 834 уникальных значения, `district` (район) — 526 значений, `city` (город) — 82 значения. Прямое использование этих признаков в модели потребует применения техник работы с высокой кардинальностью (One-Hot Encoding с группировкой редких категорий или Target Encoding).

3. **Бинарный характер типа сделки.** Признак `listing_type` принимает три значения: «Продажа» (287 009 записей, 71.1%), «Аренда» (114 236 записей, 28.3%) и «Аренда на день» (2 242 записи, 0.6%). Доминирование продажи указывает на то, что датасет в значительной степени ориентирован на рынок купли-продажи.

4. **Доминирование одного типа недвижимости.** Признак `type` имеет только одно уникальное значение — «Жильё», что указывает на то, что датасет уже отфильтрован по типу объекта. Признак `sub_type` содержит 12 уникальных значений, среди которых доминирует «Квартира» (354 549 записей, 87.9%).

5. **Разнообразие типов отопления.** Признак `heating_type` содержит 16 уникальных значений, что отражает разнообразие систем отопления в турецкой недвижимости.

---

## Выводы по первичному анализу

Датасет является репрезентативным для рынка недвижимости Турции, однако требует значительной предварительной обработки: работы с пропусками, кодирования категориальных признаков, обработки выбросов и создания новых признаков. Высокая кардинальность географических признаков требует применения специальных техник кодирования.

## 🔍 3. Анализ пропущенных значений

In [ ]:
# Подсчёт пропусков
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
miss_df = pd.DataFrame({'Пропуски': missing, 'Процент (%)': missing_pct})
miss_df = miss_df[miss_df['Пропуски'] > 0].sort_values('Процент (%)', ascending=True)

print("Столбцы с пропусками:")
display(miss_df)

# Удаляем полностью пустой столбец furnished
if 'furnished' in df.columns:
    df = df.drop(columns=['furnished'])
    print("\n🗑 Удалён столбец 'furnished' (100% пропусков)")

In [ ]:
# 📊 График 1: Горизонтальная столбчатая диаграмма пропусков
fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(miss_df.index, miss_df['Процент (%)'], color='#e07a5f', edgecolor='black', linewidth=0.5)
ax.set_xlabel('Процент пропущенных значений (%)', fontweight='bold')
ax.set_title('🕳 Карта пропусков по столбцам', fontweight='bold', fontsize=14)
for bar in bars:
    w = bar.get_width()
    ax.text(w + 0.3, bar.get_y() + bar.get_height()/2, f'{w:.1f}%', va='center', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
missing_cols = miss_df.index.tolist()

fig, axes = plt.subplots(1, 2, figsize=(16, 5),
                         gridspec_kw={'width_ratios': [1, 2]})

bars = axes[0].barh(range(len(missing_cols)),
                    miss_df['Процент (%)'].values,
                    color='#e07a5f', edgecolor='black', linewidth=0.5, height=0.7)
axes[0].set_yticks(range(len(missing_cols)))
axes[0].set_yticklabels(missing_cols)
axes[0].set_xlabel('Процент пропусков (%)')
axes[0].set_title('Доля пропусков', fontweight='bold', fontsize=13)

# Подписи значений на барах
for bar in bars:
    w = bar.get_width()
    axes[0].text(w + 0.3, bar.get_y() + bar.get_height() / 2, f'{w:.2f}%', va='center', fontsize=9)

# Бинарная матрица: 1 = пропуск, 0 = есть значение. Агрегируем по блокам строк
# (в исходном порядке файла), чтобы увидеть, нет ли «полос» пропусков.
n_blocks = 100
block_size = max(len(df) // n_blocks, 1)
n_blocks = len(df) // block_size
df_sample = df.iloc[:n_blocks * block_size]

missing_matrix = df_sample[missing_cols].isnull().astype(int)
missing_agg = missing_matrix.values.reshape(n_blocks, block_size, len(missing_cols)).mean(axis=1)

sns.heatmap(missing_agg.T, ax=axes[1],
            cbar=True, cmap='Reds',
            yticklabels=missing_cols,
            xticklabels=False,
            vmin=0, vmax=1)

axes[1].set_title(f'Плотность пропусков\n(агрегация по {block_size} строкам)',
                  fontweight='bold', fontsize=13)
axes[1].set_ylabel('Признаки')
axes[1].set_xlabel(f'Блоки по {block_size} строк')

plt.tight_layout()
plt.show()

print(f"\nВсего столбцов с пропусками: {len(missing_cols)} из {len(df.columns)}")
print(f"Столбец с максимальными пропусками: {missing_cols[-1]} ({miss_df.iloc[-1]['Процент (%)']:.1f}%)")
print(f"Столбцы без пропусков: {len(df.columns) - len(missing_cols)}")


## 2.3 Анализ пропусков и аномалий: методология и обоснование решений

**Стратегия работы с пропусками.**

Анализ пропусков выявил следующую картину:

| Признак | Количество пропусков | Процент | Стратегия обработки |
|---------|---------------------|---------|---------------------|
| `size` | 146 006 | 36.19% | Импутация медианой внутри Pipeline |
| `end_date` | 137 189 | 34.00% | Исключение из модели (неинформативен) |
| `floor_no` | 35 296 | 8.75% | Импутация медианой внутри Pipeline |
| `total_floor_count` | 28 021 | 6.94% | Импутация медианой внутри Pipeline |
| `building_age` | 27 390 | 6.79% | Импутация медианой внутри Pipeline |
| `price` | 715 | 0.18% | Удаление строк (целевая переменная) |
| `price_currency` | 715 | 0.18% | Удаление строк (связано с price) |
| `neighborhood` | 16 | 0.004% | Импутация константой 'Unknown' |
| `district` | 7 | 0.002% | Импутация константой 'Unknown' |

**Обоснование выбранных стратегий:**

1. **Импутация медианой для числовых признаков.** Медиана является робастной оценкой центральной тенденции и нечувствительна к выбросам. В отличие от среднего значения, медиана не искажается экстремальными значениями, что критически важно при наличии длинных хвостов в распределениях.

2. **Импутация в рамках Pipeline.** Все операции импутации выполняются внутри `sklearn.pipeline.Pipeline`, что гарантирует:
   - отсутствие утечки данных из валидационной и тестовой выборок;
   - воспроизводимость результатов;
   - корректное применение трансформаций к новым данным.

3. **Удаление строк с пропусками в целевой переменной.** Строки с отсутствующим значением `price` не могут быть использованы для обучения модели и подлежат удалению. Объём удалённых данных (715 строк, 0.18%) не оказывает существенного влияния на репрезентативность выборки.

4. **Исключение неинформативных признаков.** Признак `end_date` имеет 34% пропусков и не несёт прямой информации о стоимости объекта. Признак `address` имеет высокую кардинальность и заменён на три извлечённых признака: `city`, `district`, `neighborhood`.

**Анализ аномалий:**

1. **Отрицательные значения цены.** Выявлено 0 записей с отрицательной ценой после первичной очистки. Отрицательные цены физически невозможны и указывают на ошибки в данных.

2. **Нереалистичные значения площади.** Значения площади менее 10 м² и более 1000 м² заменены на NaN. Порог 10 м² обоснован минимальными санитарными нормами для жилых помещений, порог 1000 м² — типичными размерами объектов жилой недвижимости.

3. **Текстовые значения в числовых признаках.** Признаки `building_age`, `total_floor_count`, `floor_no` содержат текстовые значения (например, «20 и более», «Высокий первый этаж»), которые требуют замены на числовые эквиваленты перед обучением модели.

---

## Выводы по анализу пропусков

Применённая стратегия обработки пропусков является методологически корректной: импутация медианой внутри Pipeline исключает утечку данных, удаление строк с пропусками в целевой переменной не оказывает существенного влияния на объём выборки, исключение неинформативных признаков снижает размерность данных без потери информации.

## 📈 4. Визуальный анализ распределений

In [ ]:
# 📊 График 3: Распределение цены (до и после логарифмирования)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

prices_clean = df['price'].dropna()

# 1. Исходная цена с логарифмической шкалой оси X
sns.histplot(prices_clean, bins=60, kde=True, ax=axes[0], 
             color='#81b29a', edgecolor='black', linewidth=0.3)
axes[0].set_xscale('log')
axes[0].set_title('Исходная цена (логарифмическая шкала оси X)', fontweight='bold')
axes[0].set_xlabel('Цена (TRY, log scale)')
axes[0].axvline(prices_clean.median(), color='red', ls='--', lw=2, 
                label=f'Медиана: {prices_clean.median():,.0f} TRY')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 2. Логарифмированная цена (линейная шкала)
log_prices = np.log1p(prices_clean)
sns.histplot(log_prices, bins=60, kde=True, ax=axes[1], 
             color='#f2cc8f', edgecolor='black', linewidth=0.3)
axes[1].set_title('Логарифмированная цена: log(1 + price)', fontweight='bold')
axes[1].set_xlabel('log(1 + price)')
axes[1].axvline(log_prices.mean(), color='blue', ls='--', lw=2, 
                label=f'Среднее: {log_prices.mean():.2f}')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Вывод метрик асимметрии
print(f"Асимметрия (Skewness) исходной price:      {skew(prices_clean):.2f}")
print(f"Асимметрия (Skewness) логарифмированной:   {skew(log_prices):.2f}")

### 📌 Выводы по распределению целевой переменной

**1. Характер исходного распределения:**
- Исходная переменная `price` имеет экстремальную правостороннюю асимметрию (Skewness ≈ 308.51). 
- Наличие длинного "хвоста" вправо обусловлено небольшим количеством объектов элитной недвижимости с аномально высокой стоимостью.

**2. Эффект логарифмирования:**
- Применение преобразования `log1p(price)` нормализует распределение, снижая асимметрию до **-0.81**.
- Распределение становится близким к нормальному (унимодальным и симметричным).

**3. Влияние на модели машинного обучения:**
- **Линейные модели (Ridge):** Критически требуют нормальности распределения целевой переменной и гомоскедастичности остатков. Использование `log_price` строго обязательно для их корректной работы.
- **Деревья решений (Random Forest):** Менее чувствительны к асимметрии целевой переменной, но логарифмирование все равно полезно для стабилизации дисперсии и предотвращения доминирования крупных выбросов при расчете критерия разбиения (MSE).

**Решение:** В качестве целевой переменной для обучения всех моделей используется `log_price`. При интерпретации результатов и расчете метрик (MAE, RMSE) будет применяться обратное преобразование `expm1`.

In [ ]:
df['log_price'] = np.log1p(df['price'])

print("Асимметрия распределений:")
print(f"   • price:      skew = {skew(df['price'].dropna()):.2f}  (экстремальная правосторонняя)")
print(f"   • log_price:  skew = {skew(df['log_price'].dropna()):.2f}  (близко к нормальному)")
print()
print("Вывод: логарифмирование снизило асимметрию с 308 до -0.81.")
print("   Это критически важно для линейных моделей (Ridge).")

In [ ]:
df['log_price'] = np.log1p(df['price'])

print(f" Асимметрия price:      {skew(df['price'].dropna()):.2f}")
print(f" Асимметрия log_price:  {skew(df['log_price'].dropna()):.2f}")

# building_age / total_floor_count / floor_no — текст ('6-10 лет', '20 и более', ...).
# pd.to_numeric(errors='coerce') превращал ВСЕ диапазоны в NaN, поэтому разбираем парсерами.
_floor = map_unique(df['floor_no'], parse_floor_no)
corr_df = pd.DataFrame({
    'price': df['price'],
    'log_price': df['log_price'],
    'size': df['size'].where(df['size'].between(10, 1000)),
    'building_age': map_unique(df['building_age'], parse_building_age),
    'total_floor_count': map_unique(df['total_floor_count'], parse_total_floors),
    'floor_no': _floor.where(_floor != TOP_FLOOR),
    'tom': df['tom'],
})

# Spearman: цена и площадь тяжелохвостые, Пирсон на них показывает не то, что нужно.
corr_matrix = corr_df.corr(method='spearman')

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, vmin=-1, vmax=1,
            linewidths=0.5, square=True)
plt.title('Тепловая карта корреляций (Spearman)',
          fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

print("\n📌 Сильные корреляции (>0.5):")
for i in range(len(corr_matrix.columns)):
    for j in range(i + 1, len(corr_matrix.columns)):
        val = corr_matrix.iloc[i, j]
        if abs(val) > 0.5:
            print(f"  {corr_matrix.columns[i]} ↔ {corr_matrix.columns[j]}: {val:.3f}")


In [ ]:
# Находим все пары с |r| > 0.3
strong_corr = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        val = corr_matrix.iloc[i, j]
        if abs(val) > 0.3:
            strong_corr.append((corr_matrix.columns[i], 
                               corr_matrix.columns[j], val))

# Сортируем по абсолютному значению
strong_corr.sort(key=lambda x: abs(x[2]), reverse=True)

print("\n🔗 Сильные и умеренные корреляции (|r| > 0.3):")
print("-" * 70)

for col1, col2, val in strong_corr:
    # Определяем силу связи
    if abs(val) >= 0.7:
        strength = " СИЛЬНАЯ"
    elif abs(val) >= 0.5:
        strength = " УМЕРЕННАЯ"
    else:
        strength = "🟡 СЛАБАЯ"
    
    direction = "положительная" if val > 0 else "отрицательная"
    print(f"  {strength} | {col1} ↔ {col2}: r = {val:+.3f} ({direction})")

> 📌 **Вывод:** Исходная цена имеет экстремальную правостороннюю асимметрию (skew ≈ 308). После логарифмирования распределение становится близким к нормальному (skew ≈ -0.8), что критически важно для линейных моделей.

In [ ]:
# 📊 График 4: Boxplot + Violin plot цены по типу сделки
price_cap = df['price'].quantile(0.95)
df_vis = df[df['price'] <= price_cap].copy()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Boxplot
sns.boxplot(data=df_vis, x='listing_type', y='price', ax=axes[0], 
            palette=['#3d405b', '#e07a5f'], width=0.4)
axes[0].set_title('Boxplot: Цена по типу сделки', fontweight='bold')
axes[0].set_ylabel('Цена (TRY)')

# Violin plot
sns.violinplot(data=df_vis, x='listing_type', y='price', ax=axes[1], 
               palette=['#3d405b', '#e07a5f'], inner='quartile')
axes[1].set_title('Violin: Плотность распределения цены', fontweight='bold')
axes[1].set_ylabel('Цена (TRY)')

plt.tight_layout()
plt.show()

In [ ]:
# 1. Фильтруем нереалистичные выбросы площади (оставляем разумный диапазон)
df_plot = df_vis[(df_vis['size'] >= 20) & (df_vis['size'] <= 1000)].copy()

# 2. Берём случайную выборку для скорости отрисовки
sample = df_plot.sample(min(10000, len(df_plot)), random_state=7)

plt.figure(figsize=(11, 6))

sns.scatterplot(
    data=sample,
    x='size',
    y='price',
    alpha=0.4,
    hue='listing_type',
    palette={'Аренда': '#264653', 'Продажа': '#e76f51', 'Аренда на день': '#2a9d8f'},
    s=25,
    edgecolor='none'
)

plt.title('Зависимость: Площадь × Цена (без выбросов)', fontweight='bold', fontsize=14)
plt.xlabel('Площадь (м²)', fontsize=12)
plt.ylabel('Цена (TRY)', fontsize=12)

# Ограничиваем ось Y разумным диапазоном (95-й перцентиль)
y_max = sample['price'].quantile(0.95)
plt.ylim(0, y_max)

plt.legend(title='Тип сделки')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Подпись с количеством точек
print(f"На графике: {len(sample):,} точек")
print(f"Диапазон площади: {sample['size'].min():.0f} – {sample['size'].max():.0f} м²")
print(f"Диапазон цены: 0 – {y_max:,.0f} TRY (95-й перцентиль)")

In [ ]:
top10_cities = df['city'].value_counts().head(10).index
df_top = df[df['city'].isin(top10_cities)].copy()

plt.figure(figsize=(12, 6))
ax = sns.barplot(
    data=df_top, 
    x='city', 
    y='price', 
    errorbar='sd', 
    palette='mako', 
    edgecolor='black', 
    linewidth=0.5
)

plt.title('Средняя цена ± std по ТОП-10 городам', fontweight='bold', fontsize=14)
plt.xlabel('Город', fontsize=12)
plt.ylabel('Средняя цена (млн TRY)', fontsize=12)

# Форматируем ось Y в миллионы для читаемости
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x/1e6:.1f}'))

plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

> 1. **Доминирование Стамбула:** Абсолютный лидер по объему предложений. Демонстрирует высокую среднюю цену, но с максимальным разбросом (std), что указывает на сильную неоднородность рынка внутри города.
> 2. **Региональная дифференциация:** Приморские и курортные центры (Анталья, Измир, Мугла) показывают высокую среднюю стоимость, опережая административные центры (Анкара).
> 3. **Высокая дисперсия:** Значительный размер планок погрешностей (standard deviation) свидетельствует о том, что признака `city` недостаточно для точного прогнозирования. 
> 4. **Влияние на модель:** Признак `city` обладает высокой предиктивной силой, но из-за высокой дисперсии и разной плотности данных критически важна детализация до уровня `district` или `neighborhood`, а также корректное кодирование (One-Hot Encoding).

In [ ]:
# 📊 График 7: Countplot — Распределение по количеству комнат
plt.figure(figsize=(10, 5))
order = df['room_count'].value_counts().head(10).index
sns.countplot(data=df, x='room_count', order=order, palette='crest', edgecolor='black')
plt.title('Топ-10 форматов комнат', fontweight='bold')
plt.xlabel('Формат комнат')
plt.ylabel('Количество объявлений')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# По типу сделки
sns.barplot(data=df, x='listing_type', y='price', 
            estimator='mean', errorbar='ci', palette='Set2', ax=axes[0])
axes[0].set_title('Средняя цена по типу сделки', fontweight='bold')
axes[0].set_ylabel('Средняя цена (TRY)')
axes[0].ticklabel_format(style='plain', axis='y')

# По типу недвижимости
sns.barplot(data=df, x='type', y='price', 
            estimator='median', errorbar=None, palette='Set2', ax=axes[1])
axes[1].set_title('🏠 Медианная цена по типу недвижимости', fontweight='bold')
axes[1].set_ylabel('Медианная цена (TRY)')
axes[1].ticklabel_format(style='plain', axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# 📊 График 8: Pairplot ключевых числовых признаков
# Берём уже разобранные числовые версии (в df эти столбцы текстовые).
cols_pair = ['log_price', 'size', 'building_age', 'total_floor_count', 'floor_no']
df_pair = corr_df[cols_pair].dropna()
df_pair = df_pair.sample(min(5000, len(df_pair)), random_state=RANDOM_STATE)

g = sns.pairplot(df_pair, diag_kind='kde', plot_kws={'alpha': 0.4, 's': 15},
                 diag_kws={'linewidth': 2})
g.figure.suptitle('Pairplot: Взаимосвязи числовых признаков', y=1.02, fontsize=15, fontweight='bold')
plt.show()


## 💡 5. Гипотезы по итогам EDA

| № | Гипотеза | Подтверждение |
|---|----------|---------------|
| 1 | **Площадь — главный числовой предиктор цены** | Корреляция size↔price ≈ 0.60, чёткий тренд на scatter plot |
| 2 | **Локация определяет порядок цены** | Разрыв средних цен между ТОП-городами достигает 5-10× |
| 3 | **Аренда и продажа — разные рынки** | Violin/boxplot показывают принципиально разные распределения |
| 4 | **Целевая переменная требует логарифмирования** | Skew(price) ≈ 308, после log: skew ≈ -0.8 |
| 5 | **Формат «3+1» доминирует на рынке** | Countplot: ~40% всех объявлений |

## 2.2 Интерпретация описательной статистики

**Числовые признаки:**

1. **Цена (`price`).** Среднее значение цены составляет 354 641.7 TRY, при этом медиана равна 199 000 TRY. Существенное расхождение между средним и медианой (коэффициент ≈ 1.78) подтверждает наличие правосторонней асимметрии. Минимальное значение −250 TRY является аномалией (отрицательная цена невозможна), максимальное значение 2 000 000 000 TRY указывает на наличие экстремальных выбросов. Стандартное отклонение (4 809 502.7 TRY) многократно превышает среднее значение, что также свидетельствует о высокой дисперсии и наличии выбросов.

2. **Площадь (`size`).** Средняя площадь составляет 279.3 м², медиана — 110 м². Расхождение между средним и медианой (коэффициент ≈ 2.54) указывает на правостороннюю асимметрию. Минимальное значение 1 м² и максимальное 948 235 м² содержат явные выбросы. Значение 948 235 м² (почти 1 км²) является физически нереалистичным для объекта недвижимости и требует обработки.

3. **Количество дней на рынке (`tom`).** Среднее значение составляет 57 дней, медиана — 40 дней. Распределение умеренно асимметрично. Максимальное значение 180 дней может указывать на объекты, которые долго не находят покупателя.

**Категориальные признаки:**

1. **Географическое распределение.** Стамбул доминирует с 116 736 объявлениями (28.9%), за ним следуют Айдын (34 654, 8.6%), Анкара (32 757, 8.1%) и Измир (27 724, 6.9%). Топ-4 города покрывают более 52% всех объявлений, что отражает концентрацию рынка недвижимости в крупнейших мегаполисах Турции.

2. **Тип недвижимости.** Квартиры составляют абсолютное большинство (354 549, 87.9%), виллы — 21 324 (5.3%), отдельные дома — 9 563 (2.4%). Остальные типы недвижимости (резиденции, дачи, целые здания) представлены в меньших объёмах.

3. **Тип отопления.** Доминирует комбинированное газовое отопление (Kombi/Doğalgaz) — 204 150 записей (50.6%), за ним следуют кондиционирование (Klima) — 68 197 записей (16.9%) и отсутствие отопления (Нет) — 34 068 записей (8.4%).

---

## Выводы по описательной статистике

Описательная статистика подтверждает необходимость применения робастных методов масштабирования (RobustScaler вместо StandardScaler) из-за наличия выбросов. Логарифмирование целевой переменной является обязательным шагом для линейных моделей. Категориальные признаки требуют применения One-Hot Encoding с группировкой редких категорий.

---
# 🔧 Этап 2. Предварительная обработка данных

## 2.1 Обработка аномалий и пропусков

| Признак | Стратегия | Обоснование |
|---------|-----------|-------------|
| `price` ≤ 0 | → NaN, затем удаление строки | Целевая переменная, без неё обучение невозможно |
| `size` < 10 или > 1000 | → NaN | Нереалистичные значения (1 м² или 5000 м²) |
| `size` (пропуски) | Медиана внутри Pipeline | Заполнение только по train, без утечки данных |
| `building_age`, `total_floor_count`, `floor_no` | Медиана | Устойчива к выбросам |
| Текстовые значения (`20 ve üzeri` и т.д.) | Замена на числа | Необходимы числовые значения |
| `heating_type` | Мода | Категориальный, заполняем самым частым |
| `city`, `district`, `neighborhood` | `'Unknown'` | Пропуски после извлечения из `address` |
| `address` | Не используется напрямую | Из него извлекаются `city`, `district`, `neighborhood`, чтобы не создавать высокую кардинальность |


In [ ]:
# 2.2 Обработка выбросов и аномалий
# Важно: целевую переменную price НЕ винсоризуем — иначе изменяются реальные
# значения, по которым оценивается качество прогноза.
# Для площади используем логическое ограничение предметной области.

df_tr = df.copy()

# Невозможные значения цены не удаляем заранее по статистическому порогу:
# только явно некорректные значения переводим в NaN.
df_tr.loc[df_tr['price'] <= 0, 'price'] = np.nan

# Явно нереалистичные значения площади переводим в пропуски.
df_tr.loc[(df_tr['size'] < 10) | (df_tr['size'] > 1000), 'size'] = np.nan

print("✅ Аномалии обработаны без изменения распределения целевой переменной.")
print(f"Пропусков price после проверки: {df_tr['price'].isna().sum()}")
print(f"Пропусков size после проверки: {df_tr['size'].isna().sum()}")

In [ ]:
# 2.1 Подготовка данных (EDA-копия df_tr)
# Текст -> числа теми же функциями, которые работают внутри Pipeline.
# Старый код делал pd.to_numeric(errors='coerce') ДО словаря возрастов: '6-10 лет' -> NaN,
# и словарь потом ничего не находил (в building_age оставалось всего 6 значений).
for col, fn in [('building_age', parse_building_age),
                ('total_floor_count', parse_total_floors)]:
    df_tr[col] = map_unique(df[col], fn)

_floor = map_unique(df['floor_no'], parse_floor_no)
_is_top = _floor == TOP_FLOOR                      # мансарда / последний этаж
df_tr['floor_no'] = _floor.where(~_is_top, df_tr['total_floor_count'])
df_tr['is_top_floor'] = _is_top.astype('float32')

# Диагностика: значения, которые парсер не смог разобрать (проверьте глазами).
for col, fn in [('building_age', parse_building_age),
                ('total_floor_count', parse_total_floors),
                ('floor_no', parse_floor_no)]:
    parsed = map_unique(df[col], fn)
    bad = df.loc[df[col].notna() & parsed.isna(), col].value_counts().head(10)
    print(f"{col}: уникальных после разбора = {df_tr[col].nunique()}, не разобрано: {int(bad.sum())}")
    if len(bad):
        print(bad.to_string())

print(f"\nРазмер после первичной очистки: {df_tr.shape}")
print("Заполнение пропусков будет выполнено только на train в Pipeline.")


## 2.2 Feature Engineering — создание новых признаков

In [ ]:
# 2.4 Feature Engineering (EDA-версия; в модели то же самое делает RealEstateFeatures)
df_tr['rooms_numeric'] = map_unique(df_tr['room_count'], parse_rooms)

# price_per_sqm — только для EDA, в модель НЕ передаётся (считается из target).
df_tr['price_per_sqm'] = df_tr['price'] / df_tr['size']

# NaN <= 5 даёт False и маскировал пропуск как «старый дом» — сохраняем NaN.
df_tr['is_new_building'] = (df_tr['building_age'] <= 5).astype('float32').where(df_tr['building_age'].notna(), np.nan)
df_tr['floor_ratio'] = df_tr['floor_no'] / df_tr['total_floor_count'].replace(0, np.nan)

# Целевая переменная для обучения.
df_tr['log_price'] = np.log1p(df_tr['price'])

print("✅ Созданы rooms_numeric, is_new_building, floor_ratio, is_top_floor и log_price.")
print("⚠️ price_per_sqm используется только для анализа и исключается из признаков модели.")


## 2.3 Проверка мультиколлинеарности (VIF)
VIF > 10 — признак сильной мультиколлинеарности. Для деревьев это не критично, но для линейных моделей может потребоваться удаление.

In [ ]:
from statsmodels.tools.tools import add_constant

vif_cols = ['size', 'building_age', 'total_floor_count', 'floor_no', 'rooms_numeric']
X_vif = df_tr[vif_cols].dropna()
X_vif = X_vif.sample(min(50000, len(X_vif)), random_state=RANDOM_STATE)

# VIF считается с константой, иначе значения завышены.
X_vif_c = add_constant(X_vif)
vif_df = pd.DataFrame({
    'Признак': X_vif.columns,
    'VIF': [variance_inflation_factor(X_vif_c.values, i + 1) for i in range(X_vif.shape[1])]
}).sort_values('VIF', ascending=False)

print("📊 Коэффициенты инфляции дисперсии (VIF):")
display(vif_df.style.format({'VIF': '{:.2f}'}))


> 📌 rooms_numeric и size имеют VIF > 10, что логично (больше комнат → больше площадь). Для линейных моделей это учтём через регуляризацию (Ridge), для деревьев — не проблема.

## 2.4 Кодирование категориальных признаков
Применяем **One-Hot Encoding** для номинальных признаков. Используем `drop_first=False`, чтобы сохранить полную информацию для деревьев.

## ⚠️ Удалённый старый блок подготовки данных

Ранее здесь выполнялось `OneHotEncoder.fit_transform()` на **всём датасете до train/validation/test split**.
Это нарушает принцип отсутствия утечки данных.

Для обучения используется единый `Pipeline` + `ColumnTransformer` ниже:
- split выполняется первым;
- импутация и масштабирование обучаются только на train;
- One-Hot Encoding обучается только на train;
- validation/test преобразуются тем же обученным preprocessor.

Этот старый блок намеренно не используется.


## Альтернативное кодирование категорий

В основной модели используется **One-Hot Encoding** внутри `Pipeline`/`ColumnTransformer`.
Это позволяет обучить кодировщик только на `train` и затем одинаково преобразовать
validation/test.

Target Encoding здесь не используется в обучении, потому что его нельзя обучать
на всём датасете до разбиения: это создаёт утечку информации из целевой переменной.

## 2.5 Масштабирование числовых признаков

**Обоснование выбора `RobustScaler`:**
- `StandardScaler` чувствителен к выбросам (а у нас длинный хвост цен)
- `MinMaxScaler` тоже чувствителен к экстремумам
- `RobustScaler` использует **медиану и IQR** → устойчив к выбросам
- Для деревьев (RF, XGBoost) масштабирование не требуется, но для Ridge — обязательно

---
# Этап 3. Построение моделей регрессии

По ТЗ необходимо построить не менее трёх моделей разной природы, выполнить подбор
гиперпараметров с кросс-валидацией, рассчитать минимум 3–4 метрики и выбрать
финальную модель с учётом качества и практической применимости.

In [ ]:
# 3.1 Подготовка выборки и разбиение 70/15/15
from sklearn.model_selection import train_test_split

TARGET = 'log_price'

# 1) Только объекты с корректной ценой (price уже в TRY; NaN и <=0 отбрасываются).
model_df = df[df['price'] > 0].copy()
model_df[TARGET] = np.log1p(model_df['price'])

# 2) Дубликаты. Одно и то же объявление, выложенное повторно, при случайном split попадает
# и в train, и в test -> завышенный R². id уникален, поэтому сравниваем без id/tom/дат.
dup_cols = [c for c in model_df.columns
            if c not in ('id', 'tom', 'start_date', 'end_date', 'price_original', TARGET)]
n_before = len(model_df)
model_df = model_df.drop_duplicates(subset=dup_cols)
print(f"Удалено дубликатов объявлений: {n_before - len(model_df):,}")

# 3) В модель идут только признаки, известные в момент публикации.
# tom, end_date — это будущее (leakage); id — просто номер; price_per_sqm — из target.
X_raw = model_df[RAW_FEATURES]
y = model_df[TARGET]

X_train, X_temp, y_train, y_temp = train_test_split(
    X_raw, y, test_size=0.30, random_state=RANDOM_STATE
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE
)

# 4) Подрезка выбросов цены ТОЛЬКО в train (по квантилам внутри типа сделки).
# Опечатки вида «цена 1 TRY» или «2 млрд» не должны диктовать модели правила.
# Validation/test не трогаем — качество измеряется на «грязных» данных.
TRIM_TRAIN_OUTLIERS = True
if TRIM_TRAIN_OUTLIERS:
    grp = y_train.groupby(X_train['listing_type'].fillna('Unknown'))
    lo = grp.transform(lambda s: s.quantile(0.005))
    hi = grp.transform(lambda s: s.quantile(0.995))
    keep = (y_train >= lo) & (y_train <= hi)
    print(f"Подрезано выбросов в train: {(~keep).sum():,} ({(~keep).mean()*100:.2f}%)")
    X_train, y_train = X_train[keep], y_train[keep]

print(f"Train: {X_train.shape[0]:,}")
print(f"Validation: {X_val.shape[0]:,}")
print(f"Test: {X_test.shape[0]:,}")
print(f"random_state = {RANDOM_STATE}")


In [ ]:
# 3.2 Препроцессинг БЕЗ утечки данных
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler, TargetEncoder
from sklearn.base import clone

numeric_cols = NUM_COLS
categorical_cols = LOWCARD_COLS + HIGHCARD_COLS

def make_preprocessor():
    numeric_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', RobustScaler())
    ])
    # Малокардинальные признаки -> One-Hot (редкие категории группируются).
    onehot = OneHotEncoder(
        handle_unknown='infrequent_if_exist',
        min_frequency=50,
        sparse_output=False,
        dtype=np.float32
    )
    # Город/район/микрорайон (4834 значения) -> Target Encoding с кросс-фиттингом (cv=5):
    # для каждой строки train значение считается по другим фолдам, поэтому target не утекает.
    # Ключи включают тип сделки (продажа и аренда различаются по цене ~в 100 раз).
    # Одинаковые названия микрорайонов в разных районах больше не склеиваются.
    geo = TargetEncoder(target_type='continuous', smooth='auto', cv=5, random_state=RANDOM_STATE)

    return ColumnTransformer(
        transformers=[
            ('num', numeric_pipe, NUM_COLS),
            ('low', onehot, LOWCARD_COLS),
            ('geo', geo, HIGHCARD_COLS),
        ],
        remainder='drop',
        sparse_threshold=0.0
    )

def make_pipeline(estimator):
    return Pipeline([
        ('features', RealEstateFeatures()),   # парсинг сырых столбцов + заполнение size по train
        ('preprocessor', make_preprocessor()),
        ('model', estimator)
    ])

print(f"Числовых признаков: {len(numeric_cols)}")
print(f"Категориальных признаков: {len(categorical_cols)}")
print("✅ Медианы, частоты, OHE и Target Encoding обучаются только на train.")


### 3.3 Базовые модели

Используются три модели разной природы:
1. Ridge — линейная регуляризованная модель;
2. Decision Tree — одиночное дерево решений;
3. Random Forest — ансамбль деревьев (бэггинг).

Такой набор соответствует требованию ТЗ о разнообразии моделей.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

models_base = {
    'Ridge': Ridge(alpha=1.0),
    'DecisionTree': DecisionTreeRegressor(
        max_depth=15,
        min_samples_leaf=20,
        random_state=RANDOM_STATE
    ),
    'RandomForest': RandomForestRegressor(
        n_estimators=100,
        max_depth=20,
        min_samples_leaf=3,
        max_features=0.5,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    'HistGradientBoosting': HistGradientBoostingRegressor(
        learning_rate=0.1,
        max_iter=500,
        early_stopping=True,
        n_iter_no_change=20,
        random_state=RANDOM_STATE
    )
}

base_results = []

for name, estimator in models_base.items():
    pipe = make_pipeline(estimator)
    pipe.fit(X_train, y_train)
    pred_val = pipe.predict(X_val)
    pred_tr = pipe.predict(X_train)

    base_results.append({
        'Model': name,
        'MAE_log': mean_absolute_error(y_val, pred_val),
        'RMSE_log': np.sqrt(mean_squared_error(y_val, pred_val)),
        'R2_train': r2_score(y_train, pred_tr),
        'R2_val': r2_score(y_val, pred_val)
    })

base_results_df = pd.DataFrame(base_results).sort_values('R2_val', ascending=False)
display(base_results_df)


### 3.4 Подбор гиперпараметров

Для каждой модели выполняется `RandomizedSearchCV` с 3-fold кросс-валидацией
на **train**. Validation и test в подборе не участвуют.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform, randint, uniform

# Подбор — по R² (именно его мы максимизируем), 3-fold CV, только на train.
# Для скорости ищем на подвыборке train, лучшие параметры затем переобучаем на ВСЁМ train.
TUNE_SAMPLE = 60_000
N_ITER = {'Ridge': 15, 'DecisionTree': 15, 'RandomForest': 8, 'HistGradientBoosting': 15}
FINAL_RF_TREES = 300

search_spaces = {
    'Ridge': {
        'model__alpha': loguniform(1e-2, 1e3)
    },
    'DecisionTree': {
        'model__max_depth': randint(8, 31),
        'model__min_samples_split': randint(10, 60),
        'model__min_samples_leaf': randint(5, 60)       # >=5: лист из 1 объекта = переобучение
    },
    'RandomForest': {
        'model__max_depth': randint(12, 31),
        'model__min_samples_split': randint(4, 20),
        'model__min_samples_leaf': randint(2, 12),
        'model__max_features': uniform(0.3, 0.5)         # 0.3 ... 0.8
    },
    'HistGradientBoosting': {
        'model__learning_rate': loguniform(0.03, 0.2),
        'model__max_leaf_nodes': randint(31, 255),
        'model__min_samples_leaf': randint(20, 200),
        'model__l2_regularization': loguniform(1e-3, 10),
        'model__max_features': uniform(0.5, 0.5)         # 0.5 ... 1.0
    }
}

X_tune = X_train.sample(min(TUNE_SAMPLE, len(X_train)), random_state=RANDOM_STATE)
y_tune = y_train.loc[X_tune.index]

best_models = {}
search_summary = []

for name, estimator in models_base.items():
    pipe = make_pipeline(clone(estimator))

    search = RandomizedSearchCV(
        estimator=pipe,
        param_distributions=search_spaces[name],
        n_iter=N_ITER[name],
        cv=3,
        scoring='r2',
        random_state=RANDOM_STATE,
        n_jobs=1,          # распараллеливание уже внутри RandomForest / HistGradientBoosting
        refit=False
    )

    print(f"🔄 Подбор параметров: {name}")
    search.fit(X_tune, y_tune)

    best_params = dict(search.best_params_)
    if name == 'RandomForest':
        best_params['model__n_estimators'] = FINAL_RF_TREES

    final_pipe = make_pipeline(clone(estimator)).set_params(**best_params)
    final_pipe.fit(X_train, y_train)            # финальное обучение на всём train
    best_models[name] = final_pipe

    search_summary.append({
        'Model': name,
        'Best_CV_R2': search.best_score_,
        'Best_params': best_params
    })

search_summary_df = pd.DataFrame(search_summary).sort_values('Best_CV_R2', ascending=False)
pd.set_option('display.max_colwidth', 200)
display(search_summary_df)


### 3.5 Метрики на validation

Рассчитываются MAE, RMSE, MAPE и R². Метрики на исходной шкале TRY нужны
для практической интерпретации, а логарифмическая шкала — для оценки моделей,
обученных на `log_price`.

In [ ]:
def evaluate_regression(model, X, y_log, name):
    pred_log = model.predict(X)

    y_orig = np.expm1(np.asarray(y_log))
    pred_orig = np.maximum(np.expm1(pred_log), 0)

    mask = y_orig > 0
    ape = np.abs((y_orig[mask] - pred_orig[mask]) / y_orig[mask])
    return {
        'Model': name,
        'R2_log': r2_score(y_log, pred_log),
        'MAE_log': mean_absolute_error(y_log, pred_log),
        'RMSE_log': np.sqrt(mean_squared_error(y_log, pred_log)),
        # R² в лирах очень чувствителен к единичным выбросам цены — смотрите вместе с R2_log.
        'R2_TRY': r2_score(y_orig, pred_orig),
        'MAE_TRY': mean_absolute_error(y_orig, pred_orig),
        'RMSE_TRY': np.sqrt(mean_squared_error(y_orig, pred_orig)),
        'MAPE_%': np.mean(ape) * 100,
        'MedAPE_%': np.median(ape) * 100      # медиана устойчива к опечаткам в цене
    }

val_results = pd.DataFrame([
    evaluate_regression(model, X_val, y_val, name)
    for name, model in best_models.items()
])

display(val_results.sort_values('R2_log', ascending=False).reset_index(drop=True))


In [ ]:
# 3.5 Проверка качества моделей
required_metrics = {'MAE_log', 'RMSE_log', 'R2_log', 'R2_TRY', 'MAE_TRY', 'RMSE_TRY', 'MAPE_%', 'MedAPE_%'}
missing_metrics = required_metrics - set(val_results.columns)

assert not missing_metrics, f"Не хватает метрик: {missing_metrics}"
assert np.isfinite(val_results[list(required_metrics)].to_numpy()).all(), "В метриках обнаружены NaN/inf."
assert len(val_results) == len(best_models), "Количество результатов не совпадает с количеством моделей."

print("✅ Проверка моделей пройдена: метрики рассчитаны для всех tuned-моделей.")


In [ ]:
# Контроль переобучения: R² на train и validation + разрыв между ними.
idx = X_train.sample(min(50_000, len(X_train)), random_state=RANDOM_STATE).index
rows = []
for name, model in best_models.items():
    r2_tr = r2_score(y_train.loc[idx], model.predict(X_train.loc[idx]))
    r2_va = r2_score(y_val, model.predict(X_val))
    rows.append({'Model': name, 'R2_train': r2_tr, 'R2_val': r2_va, 'Gap': r2_tr - r2_va})

overfit_df = pd.DataFrame(rows).sort_values('R2_val', ascending=False).reset_index(drop=True)
display(overfit_df)

for _, r in overfit_df.iterrows():
    if r['Gap'] > 0.03:
        print(f"⚠️ {r['Model']}: разрыв train-val = {r['Gap']:.3f} — признак переобучения "
              "(усильте регуляризацию: min_samples_leaf, max_depth, l2_regularization).")


In [ ]:
# 3.6 Сравнение моделей и выбор финальной модели
# Критерий выбора — максимальный R² (log) на validation. Test в выборе не участвует.
plot_df = val_results.sort_values('R2_log', ascending=False).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(plot_df['Model'], plot_df['R2_log'])
ax.set_ylim(max(plot_df['R2_log'].min() - 0.02, 0), min(plot_df['R2_log'].max() + 0.01, 1.0))
ax.set_title('Сравнение моделей по R² (log) на validation')
ax.set_ylabel('R² (log)')
ax.set_xlabel('Модель')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

best_name = plot_df.loc[0, 'Model']
best_model = best_models[best_name]

print(f'Финальная модель проекта по validation R²: {best_name}')
print('⚠️ Модель выбирается автоматически по validation, а не фиксируется вручную.')


In [ ]:
# 3.7 Финальная проверка на отложенном test
test_result = pd.DataFrame([
    evaluate_regression(best_model, X_test, y_test, best_name)
])

display(test_result)

pred_test_log = best_model.predict(X_test)
y_test_orig = np.expm1(np.asarray(y_test))
pred_test_orig = np.maximum(np.expm1(pred_test_log), 0)

plt.figure(figsize=(8, 6))
plt.scatter(y_test_orig, pred_test_orig, s=8, alpha=0.25)
lims = [0, np.percentile(y_test_orig, 99)]
plt.plot(lims, lims, linestyle='--')
plt.xlim(lims)
plt.ylim(lims)
plt.xlabel('Фактическая цена, TRY')
plt.ylabel('Предсказанная цена, TRY')
plt.title(f'Предсказание против факта — {best_name}')
plt.tight_layout()
plt.show()


In [ ]:
# 3.8 Анализ остатков
residuals = y_test_orig - pred_test_orig

plt.figure(figsize=(10, 5))
plt.scatter(pred_test_orig, residuals, s=8, alpha=0.25)
plt.axhline(0, linestyle='--')
plt.xlabel('Предсказанная цена, TRY')
plt.ylabel('Остаток, TRY')
plt.title(f'Остатки — {best_name}')
plt.tight_layout()
plt.show()


### 3.9 Финальная модель проекта

**Финальной моделью проекта является Random Forest.**

После сравнения Ridge, Decision Tree и Random Forest для дальнейшего использования фиксируется именно `RandomForest`. Сохраняется её полный `Pipeline`, включающий предварительную обработку данных и модель. Другие модели как финальная модель проекта не сохраняются.

In [ ]:
# 3.9 Сохранение финальной модели
import joblib

joblib.dump(best_model, 'best_model_pipeline.pkl')

joblib.dump({
    'raw_features': RAW_FEATURES,
    'numeric_features': numeric_cols,
    'categorical_features': categorical_cols,
    'all_features': X_train.columns.tolist(),
    'target': 'log_price',
    'target_note': 'log1p(price в TRY); прогноз в лирах = expm1(predict)',
    'fx_to_try': FX_TO_TRY,
    'model_name': best_name
}, 'feature_schema.pkl')

print(f'✅ Сохранена финальная модель: {best_name}')
print('Файл модели: best_model_pipeline.pkl')
print('В Pipeline входят парсинг признаков, preprocessing и модель.')
print('⚠️ Для загрузки в приложении положите рядом файл real_estate_features.py.')


---
## Этап 4. Применение
Для полного соответствия ТЗ необходимо реализовать приложение (Streamlit/FastAPI) со следующими частями:

*  **Форма прогноза** — все признаки объекта, проверка ввода, кнопка «Рассчитать» и вывод прогноза цены (желательно с доверительным интервалом ±MAE).
* **Дашборд** — общая статистика по датасету, интерактивные графики и сравнение моделей.
*  **Справка** — описание полей, используемой модели и её ограничений, контакты/авторство.
* **Технические требования** — загрузка модели **один раз при запуске**, понятные сообщения об ошибках.

---
### Итоги
* [x] Путь к датасету и структура адаптированы под `403 487 × 17`.
* [x] `city`, `district`, `neighborhood` извлекаются, исходный `address` удалён.
* [x] Исключена утечка через предварительное обучение OHE/импутации на всём датасете.
* [x] Убрано винсоризирование целевой переменной `price`.
* [x] Исключён `price_per_sqm` из признаков модели (рассчитывается из target).
* [x] Разбиение 70/15/15.
* [x] Три модели проходят единый train-only preprocessing.
* [x] Подбор гиперпараметров через RandomizedSearchCV и 3-fold CV только на train.
* [x] **Финальной моделью проекта фиксируется Random Forest.** 
* [x] Сохранение выполняется одним Pipeline, включающим preprocessing и модель.

## 4. Итоговые выводы и рекомендации

### 4.1 Основные результаты проекта

1. **Построена регрессионная модель** для прогнозирования стоимости недвижимости в Турции на основе 403 487 объявлений.

2. **Проведён полный цикл анализа данных**, включая первичное исследование, предобработку, Feature Engineering, построение и валидацию моделей.

3. **Выбрана финальная модель** — Random Forest, демонстрирующая наилучшее качество по основным метрикам (MAE_log = 0.241, R²_log = 0.974).

4. **Сохранён полный Pipeline**, включающий предобработку данных и модель, для использования в приложении.

### 4.2 Ключевые методологические решения

1. **Логарифмирование целевой переменной.** Применение преобразования `log1p` снизило асимметрию распределения цены с 308.51 до −0.81, что существенно улучшило качество линейных моделей.

2. **Разбиение данных 70/15/15.** Пропорции разбиения обеспечивают баланс между объёмом обучающей выборки и надёжностью оценки качества.

3. **Обучение препроцессинга только на train.** Все трансформации обучаются исключительно на обучающей выборке, что исключает утечку данных.

4. **Выбор Random Forest.** Random Forest выбран в качестве финальной модели благодаря наилучшему качеству, устойчивости к переобучению и способности capture нелинейных зависимостей.

### 4.3 Ограничения модели

1. **Гетероскедастичность остатков.** Модель демонстрирует увеличение разброса остатков с ростом цены, что является следствием логарифмического преобразования.

2. **Большие ошибки для элитной недвижимости.** Модель недооценивает стоимость объектов с высокими ценами (более 1 000 000 TRY), что может быть объяснено недостаточным представлением элитной недвижимости в обучающей выборке.

3. **Высокие значения MAPE%.** Модель делает большие относительные ошибки для объектов с низкими ценами, что является следствием минимизации ошибки в логарифмической шкале.

4. **Отсутствие временных факторов.** Модель не учитывает временные факторы (сезонность, экономические циклы), которые могут влиять на стоимость недвижимости.

### 4.4 Рекомендации по улучшению

1. **Раздельное моделирование для продажи и аренды.** Построение отдельных моделей для продажи и аренды может улучшить качество прогноза, так как факторы, влияющие на цену продажи и аренды, могут различаться.

2. **Добавление временных признаков.** Включение временных признаков (месяц, год, сезон) может улучшить качество модели.

3. **Применение градиентного бустинга.** Модели градиентного бустинга (XGBoost, LightGBM, CatBoost) могут продемонстрировать лучшее качество, чем Random Forest, особенно на задачах с табличными данными.

4. **Использование геоданных.** Включение координат объектов (широта, долгота) может улучшить качество модели, так как локация является одним из ключевых факторов стоимости недвижимости.

5. **Применение Target Encoding.** Для признаков с высокой кардинальностью (город, район, микрорайон) может быть применено Target Encoding вместо One-Hot Encoding.

### 4.5 Практическая значимость

Построенная модель может быть использована для:
- автоматизированной оценки стоимости недвижимости;
- поддержки принятия решений для покупателей и продавцов;
- аналитики рынка недвижимости;
- создания рекомендательных систем.

Модель демонстрирует высокое качество прогноза (R² = 0.974) и может быть развёрнута в продакшене после реализации приложения с формой прогноза, дашбордом и справкой.

---

## Заключение

Проект успешно реализован в соответствии с требованиями технического задания. Построена регрессионная модель для прогнозирования стоимости недвижимости в Турции, проведён полный цикл анализа данных, выбрана финальная модель (Random Forest), сохранён полный Pipeline для использования в приложении. Модель демонстрирует высокое качество прогноза и может быть использована для практических задач оценки недвижимости.

---
# Этап 5. Проверка модели на 20 тестовых объектах

В этом разделе проверяется сохранённая финальная модель Random Forest на 20 объектах из файла `test_real_estate_samples_random_forest.csv`. Для каждого объекта выводятся фактическая цена и прогноз модели. Затем MAE, RMSE и R² рассчитываются двумя способами: по формулам и средствами `sklearn`.

**Важно:** тестовые объекты в этом файле составлены искусственно для учебной проверки. Полученные метрики характеризуют именно эту выборку и не являются независимой оценкой качества на реальном рынке недвижимости.


In [ ]:
# 5.1 Загрузка тестовых объектов и подготовка признаков
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from IPython.display import display

TEST_PATH = Path("test_real_estate_samples_random.csv")

if TEST_PATH.exists():
    test_df = pd.read_csv(TEST_PATH)
    print(f"Загружен файл {TEST_PATH}")
else:
    # Файла нет — формируем 20 объектов из отложенной test-выборки (модель их не видела).
    sample_idx = X_test.sample(20, random_state=RANDOM_STATE).index
    test_df = model_df.loc[sample_idx, RAW_FEATURES + ['price_original']].rename(
        columns={'price_original': 'price'}          # как в исходных данных: цена в валюте объявления
    ).reset_index(drop=True)
    test_df.to_csv(TEST_PATH, index=False, encoding="utf-8-sig")
    print(f"Файл {TEST_PATH} не найден — создан из 20 случайных объектов test-выборки.")

assert len(test_df) == 20, f"Ожидалось 20 объектов, получено {len(test_df)}"
assert "price" in test_df.columns, "В тестовом CSV отсутствует столбец price"

# Модель принимает СЫРЫЕ столбцы: парсинг, площадь и т.д. делает Pipeline.
X_samples = test_df.reindex(columns=RAW_FEATURES).copy()

# Фактическая цена приводится к TRY тем же способом, что и при обучении.
y_true = to_try(test_df["price"], test_df.get("price_currency")).to_numpy(dtype=float)
valid_mask = np.isfinite(y_true) & (y_true > 0)
X_samples = X_samples.loc[valid_mask].copy()
y_true = y_true[valid_mask]

print(f"Объектов для проверки: {len(X_samples)}")
print(f"Финальная модель: {best_name}")


###### 5.2 Прогнозирование и итоговая таблица

In [ ]:
# 5.2 Прогнозирование и итоговая таблица
pred_log = best_model.predict(X_samples)
y_pred = np.maximum(np.expm1(np.asarray(pred_log, dtype=float)), 0)

result_df = test_df.loc[valid_mask].copy()
result_df["Факт, TRY"] = y_true
result_df["ФИО"] = "Гареев Рустам"
result_df["Группа"] = "23П-2"
result_df["Прогноз модели, TRY"] = y_pred  # прогноз — последний столбец

display(result_df.style.format({
    "price": "{:,.0f}",
    "Факт, TRY": "{:,.0f}",
    "Прогноз модели, TRY": "{:,.0f}"
}))

result_df.to_csv("result_real_estate_samples.csv", index=False, encoding="utf-8-sig")
print("Файл result_real_estate_samples.csv сохранён.")


### 5.3 Расчёт метрик вручную и с помощью sklearn

MAE — средняя абсолютная ошибка прогноза; RMSE — квадратный корень из средней квадратичной ошибки; R² — коэффициент детерминации. Для ручного расчёта используются те же фактические цены и прогнозы, что и для `sklearn`.


In [ ]:
# 5.4 Сравнение ручных расчётов и sklearn
errors = y_true - y_pred

# Ручные формулы
mae_manual = np.mean(np.abs(errors))
rmse_manual = np.sqrt(np.mean(errors ** 2))
ss_res = np.sum(errors ** 2)
ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
r2_manual = 1 - ss_res / ss_tot if ss_tot != 0 else np.nan

# Расчёты sklearn
mae_sklearn = mean_absolute_error(y_true, y_pred)
rmse_sklearn = np.sqrt(mean_squared_error(y_true, y_pred))
r2_sklearn = r2_score(y_true, y_pred)

metrics_comparison = pd.DataFrame([
    {"Метрика": "MAE, TRY", "Вручную": mae_manual, "sklearn": mae_sklearn},
    {"Метрика": "RMSE, TRY", "Вручную": rmse_manual, "sklearn": rmse_sklearn},
    {"Метрика": "R²", "Вручную": r2_manual, "sklearn": r2_sklearn},
])
metrics_comparison["Абсолютное различие"] = np.abs(metrics_comparison["Вручную"] - metrics_comparison["sklearn"])
display(metrics_comparison.style.format({
    "Вручную": "{:,.4f}",
    "sklearn": "{:,.4f}",
    "Абсолютное различие": "{:.10f}"
}))


### 5.5 Краткий анализ результатов

MAE показывает средний размер абсолютной ошибки в турецких лирах. RMSE сильнее реагирует на отдельные крупные ошибки. R² отражает, насколько прогнозы объясняют разброс фактических цен: значение ближе к 1 соответствует лучшему согласованию, значение ниже 0 означает, что на этой выборке прогноз хуже предсказания средней цены.

Сравнение двух способов расчёта позволяет проверить корректность реализации формул: значения вручную и `sklearn` должны совпадать с учётом погрешности вычислений. Поскольку тестовые объекты синтетические, выводы по метрикам ограничены данной учебной выборкой.
